# EEG Fast Fourier TransformationS (FFTS)



|Previous Notebook|Current Notebook|Next Notebook|
|:-:|:-:|:-:|
|[5.frame_alignment.ipynb](./5.frame_alignment.ipynb)|<i>6.eeg_ffts.ipynb</i>|-|

In the previous notebook, [5.frame_alignment.ipynb](./5.frame_alignment.ipynb), we corrected the EEG timing for simulations and actually created a mapper dataframe to map frames to milliseconds per participant. In this notebook, we'll actually attempt to analyze EEG data by transforming the data into Power Spectral Density (PSD) charts.

## Step 0: Imports

In [3]:
!pip install plotly

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.8/9.8 MB 12.7 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 418.1/418.1 kB 11.2 MB/s eta 0:00:00


In [17]:
import numpy as np
import pandas as pd
import os

from scipy.signal import spectrogram, get_window
import plotly.subplots as sp
import plotly.graph_objects as go

#from scipy.signal.windows import hamming

_CHANNELS = ['AF7', 'AF8', 'TP9', 'TP10']
_BANDS = {
    "delta": (1, 4),
    "theta": (4, 8),
    "alpha": (9, 13),
    "beta": (13, 30),
    "gamma": (30, 200)
}

## Step 1: Calculating PSDs

For each EEG channel (AF7, AF8, TP9, and TP10), we can convert the time-series EEG signal data into a Power Spectral Density (PSD) form. The PSD informs us of which frequencies are strongest in the signal at a given timestamp.

In a typical scenario, we would use a Fast Fourier Transformation (FFT). However, I want to replicate the operations used by the Muse system to calculate the PSD. Let's create a helper function to do this, which uses `scipy.signal.spectrogram` to calculate the PSD. We also want to be cognizant of 

In [18]:
# Define some primitives
_DATA_DIR = os.path.join('.','data2')
_EXAMPLE_PARTICIPANT = 'P4_Muse2'
_EXAMPLE_TRIAL = 1

# Read trial data
tdir = os.path.join(_DATA_DIR, _EXAMPLE_PARTICIPANT, f"{_EXAMPLE_TRIAL}", "simulation")
raw_eeg = pd.read_csv(os.path.join(tdir, 'cor_eeg_raw_framed.csv'))

# Calculate the PSD from `compute_muse_psd`
psd = compute_muse_psd(raw_eeg, _CHANNELS)

# Plot
plot_muse_psd(psd)


In [10]:
def compute_muse_psd(df, channels, sample_rate=220, window_size=256, slide_rate=22):
    
    # Get the hamming window class with the defined window size
    win = get_window('hamming', window_size)

    # Intiialize outputes
    psd = {}
    freqs = None
    times = None
    
    # For each channel, we:
    for ch in channels:
        # use `spectrogram` helper to calculate properties of the raw EEG signal in that channel
        f, t, Sxx = spectrogram(
            df[ch].values,
            fs=sample_rate,
            window=win,
            nperseg=window_size,
            noverlap=window_size - slide_rate,
            scaling='density',
            mode='psd'
        )
        # Convert to decibels (dB)
        Sxx_dB = 10 * np.log10(Sxx + 1e-12)
        
        # Add our responses to out outputs. We only do this once.
        psd[ch] = Sxx_dB
        if freqs is None:
            freqs = f
            times = t
    
    # Return the three data extracted from all eeg channels
    return EEG(freqs, times, psd)

In [15]:
def plot_muse_psd(eeg_psd:EEG):
    # Extract the necessary 
    freqs = eeg_psd.freqs
    times = eeg_psd.times
    psd = eeg_psd.psd
    channels = eeg_psd.channels

    fig = sp.make_subplots(rows=2, cols=2, subplot_titles=channels)

    for i, ch in enumerate(channels):
        heatmap = go.Heatmap(
            z=psd[ch],
            x=times,
            y=freqs,
            colorscale='Viridis',
            colorbar=dict(title='Power (dB)'),
            zmin=-40, zmax=20
        )
        row, col = divmod(i, 2)
        fig.add_trace(heatmap, row=row+1, col=col+1)
        fig.update_yaxes(title_text="Freq (Hz)", row=row+1, col=col+1)
        fig.update_xaxes(title_text="Time (s)", row=row+1, col=col+1)

    fig.update_layout(
        title="Muse EEG Power Spectral Density (PSD)",
        height=900,
        width=1000
    )

    fig.show()